In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from scipy.stats import randint, uniform, loguniform

RANDOM_STATE = 42


In [2]:
train_path = "train.csv"
test_path  = "test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print(train_df.shape, test_df.shape)
train_df.head()


(15000, 10) (10000, 9)


,id,Sex,Length,Diameter,Height,Weight,Shucked Weight,Viscera Weight,Shell Weight,Age
0,0,F,1.6000,1.2500,0.5750,43.700754,15.025235,10.262519,13.324265,17.0
1,1,M,1.5750,1.1625,0.3750,32.417653,12.927372,6.208540,8.930093,20.0
2,2,F,1.6125,1.2750,0.4375,34.373769,14.784264,9.142714,10.772810,10.0
3,3,M,1.2250,0.9875,0.3250,18.455524,8.037083,3.926406,4.819415,9.0
4,4,I,1.3750,1.0250,0.3875,22.736299,9.879801,5.513978,6.945627,10.0


In [3]:
TARGET = "Age"
ID_COL = "id"

assert TARGET in train_df.columns, "Donot find target in train!"
assert TARGET not in test_df.columns, "Donot find target in test!"

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET].astype(float)

print("Train columns:", list(train_df.columns))
print("Missing values (train):\n", train_df.isna().sum())
print("dtypes:\n", train_df.dtypes)


Train columns: ['id', 'Sex', 'Length', 'Diameter', 'Height', 'Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight', 'Age']
Missing values (train):
 id                0
Sex               0
Length            0
Diameter          0
Height            0
Weight            0
Shucked Weight    0
Viscera Weight    0
Shell Weight      0
Age               0
dtype: int64
dtypes:
 id                  int64
Sex                object
Length            float64
Diameter          float64
Height            float64
Weight            float64
Shucked Weight    float64
Viscera Weight    float64
Shell Weight      float64
Age               float64
dtype: object


In [6]:
drop_cols = [ID_COL] if ID_COL in X.columns else []
X = X.drop(columns=drop_cols)
X_test = test_df.drop(columns=drop_cols)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical cols:", cat_cols)
print("Numeric cols:", num_cols)


Categorical cols: ['Sex']
Numeric cols: ['Length', 'Diameter', 'Height', 'Weight', 'Shucked Weight', 'Viscera Weight', 'Shell Weight']


In [7]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)


TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'